# 資料讀取及合併

# 1. 資料讀取 function

* 資料內含有以下三個 section：
    * `Customer Service Interruptions`：客戶服務中斷
    * `Transmission Line Interruptions`：輸電線路中斷（本研究目標）
    * `Transformer Interruptions`：變壓器中斷（與輸電線路無關，排除）

* 僅保留 `Transmission Line Interruptions` 的資料，讀取範圍限制在該 section 內，避免將 `Transformer Interruptions` 的資料一併讀入

In [4]:
import pandas as pd
import os

DATA_DIR = "BPA_DATA"
# 找到 "Transmission Line Interruptions" 的起始與結束列號
def find_transmission_row(path):
    # 讀取 excel 檔案
    df_raw = pd.read_excel(path, header=None)

    t_start = None
    t_end = None

    # 逐行檢查，找到 Transmission Line Interruptions 的起始與下一個 section 的列號
    for i, row in df_raw.iterrows():
        # 讀取第一個欄位
        val = str(row.iloc[0])
        # 搜尋 Transmission Line Interruptions 起始點 idx
        if "Transmission Line Interruptions" in val:
            t_start = i
        # 搜尋 Transmission Line Interruptions 終點 idx
        elif t_start is not None and "Interruptions" in val:
            t_end = i
            break

    return t_start, t_end

# 讀取單一檔案，擷取 Transmission Line Interruptions 部分
def load_file(path):
    # 獲取資料年份
    year = int(os.path.basename(path).replace("OutagesCY", "").split(".")[0])

    # 找到輸電資料起始與結束列
    t_start, t_end = find_transmission_row(path)

    # 計算需讀取的列數
    nrows = (t_end - t_start - 2) if t_end else None

    # 讀取目標列以下資料
    df = pd.read_excel(path, header=t_start + 1, nrows=nrows)

    # 移除空欄位
    df = df[[c for c in df.columns if not str(c).startswith("Unnamed")]]

    # 新增欄位顯示資料的年份
    df["year"] = year

    return df

# 2. 讀取所有原始資料

* 讀取 BPA_DATA 資料夾內所有年份檔案並合併

In [5]:
# 目標檔案為檔案夾內開頭為 OutagesCY 者
files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("OutagesCY")])

# 逐筆取出目標檔案 dataframe
dfs = [load_file(os.path.join(DATA_DIR, f)) for f in files]

# 合併資料 
df_raw = pd.concat(dfs, ignore_index=True)

# Check
df_raw.shape

FileNotFoundError: [Errno 2] No such file or directory: 'BPA_DATA\\OutagesCY2025.xlsx'

# 儲存資料合併成果

In [ ]:
# 建立輸出資料夾
output_dir = "raw_data"
os.makedirs(output_dir, exist_ok=True)

# 儲存合併結果
df_raw.to_csv(os.path.join(output_dir, "bpa_data_merged.csv"), index=False)